# 06 — Persistência, carregamento e inferência

`state_dict` + configuração + pré-processamento em `.pt`, carregamento com `weights_only=True` e `map_location='cpu'`, verificação de fidelidade e inferência em script independente.

In [1]:
import json, subprocess, sys
sys.path.append('..')
import pandas as pd
from src.persistencia import (carregar_checkpoint, reconstruir_modelo,
                                                     verificar_fidelidade)
from src.config import ROOT
linhas = []
for nome in ['regressao', 'classificacao']:
    ck = carregar_checkpoint(ROOT / 'models' / f'modelo_{nome}.pt')
    m = reconstruir_modelo(ck)
    fid = verificar_fidelidade(nome)
    print(nome, '| parametros:', m.n_parametros, '| fidelidade:', fid)
    linhas.append({'problema': nome, 'parametros': m.n_parametros, **fid})
pd.DataFrame(linhas).to_csv(ROOT / 'reports/tables/06_fidelidade_modelos.csv', index=False)

regressao | parametros: 9729 | fidelidade: {'diferenca_maxima_absoluta': 2.384185791015625e-07, 'allclose': True, 'n_amostras': 8000}


classificacao | parametros: 1971 | fidelidade: {'diferenca_maxima_absoluta': 1.9073486328125e-06, 'allclose': True, 'n_amostras': 8000}


## Inferência em ambiente limpo (script separado)

In [2]:
entrada = ROOT / 'data/processed/dataset_classification.csv'
cmd = [sys.executable, str(ROOT / 'src/inference.py'),
       '--modelo', str(ROOT / 'models/modelo_classificacao.pt'),
       '--entrada', str(entrada), '--n', '5']
saida = subprocess.check_output(cmd, text=True)
print(saida)

{
  "problema": "classificacao",
  "classes": [
    "Low",
    "Moderate",
    "High"
  ],
  "limiar": 0.0,
  "previsoes": [
    "High",
    "High",
    "Moderate",
    "Low",
    "Low"
  ],
  "confianca": [
    0.914886,
    0.598802,
    0.772737,
    0.999314,
    0.672128
  ]
}

